# Q01 · Monte Carlo starter notebook

Quant track, session Q01 (Tue 24 Nov 2026).

Run each cell top to bottom. Cells marked TODO are yours: change the code, rerun, and write your answer in the markdown cell below them.

What you'll see today:
1. The law of large numbers with coin flips
2. Monte Carlo estimation of pi, and why its error shrinks like 1/sqrt(n)
3. Random walks, and why variance grows linearly with time
4. Your first simulated stock price paths (geometric Brownian motion)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=24)   # fixed seed so your results are reproducible
plt.rcParams["figure.figsize"] = (9, 4)
print("numpy", np.__version__)

## 1. Law of large numbers

Flip a fair coin n times. The running fraction of heads should settle near 0.5, and the wobble should shrink as n grows.

In [ ]:
n = 10_000
p = 0.5
flips = rng.random(n) < p
running = np.cumsum(flips) / np.arange(1, n + 1)

plt.plot(running, lw=1)
plt.axhline(p, color="k", ls="--", lw=1)
plt.xscale("log")
plt.xlabel("number of flips (log scale)")
plt.ylabel("fraction of heads")
plt.title("Running average converges to p")
plt.show()
print("final estimate:", running[-1])

TODO 1. Set `p = 0.53` (a slightly biased coin, like a small trading edge). Roughly how many flips do you need before the running average is clearly separated from 0.5? Write your answer here:

> your answer

## 2. Monte Carlo estimate of pi

Throw random points into the unit square. The fraction landing inside the quarter circle is pi/4.

In [ ]:
def mc_pi(n, rng):
    x = rng.random(n)
    y = rng.random(n)
    return 4 * np.mean(x * x + y * y <= 1.0)

ns = np.logspace(2, 7, 11).astype(int)
errors = []
for n in ns:
    reps = [abs(mc_pi(n, rng) - np.pi) for _ in range(20)]
    errors.append(np.mean(reps))
    print(f"n={n:>10,d}  mean abs error={errors[-1]:.5f}")

plt.loglog(ns, errors, "o-", label="Monte Carlo error")
plt.loglog(ns, errors[0] * np.sqrt(ns[0] / ns), "--", label="1/sqrt(n) reference")
plt.xlabel("n points")
plt.ylabel("mean |error|")
plt.legend()
plt.title("Monte Carlo error shrinks like 1/sqrt(n)")
plt.show()

TODO 2. From the plot: to get one more correct decimal place (10x smaller error), how many more points do you need? What does that mean for pricing options with Monte Carlo later (session Q17)?

> your answer

## 3. Random walks

Each step is +1 or -1 with equal probability. Simulate many walks and watch how spread out they get over time.

In [ ]:
walks, steps = 2_000, 250   # 250 is roughly the number of trading days in a year
moves = rng.choice([-1, 1], size=(walks, steps))
paths = np.cumsum(moves, axis=1)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(paths[:30].T, lw=0.8)
ax[0].set_title("30 random walks")
ax[0].set_xlabel("step")

var_t = paths.var(axis=0)
ax[1].plot(np.arange(1, steps + 1), var_t, label="simulated variance")
ax[1].plot(np.arange(1, steps + 1), np.arange(1, steps + 1), "--", label="t (theory)")
ax[1].set_title("Variance grows linearly with time")
ax[1].set_xlabel("step")
ax[1].legend()
plt.show()

TODO 3. If variance grows like t, how does the standard deviation grow? So if a stock's daily volatility is 1%, what is its rough annual volatility over 250 trading days? (This is the square-root-of-time rule you'll use constantly.)

> your answer

## 4. Your first simulated stock prices (GBM)

Geometric Brownian motion is the classic model behind Black-Scholes:

S(t+dt) = S(t) · exp((mu - sigma²/2)·dt + sigma·sqrt(dt)·Z), with Z ~ N(0, 1)

The numbers below are rough Nifty-like assumptions (12% drift, 18% volatility). In Q02 and Q14 you'll estimate them from real data.

In [ ]:
S0, mu, sigma = 100.0, 0.12, 0.18
T, steps, n_paths = 1.0, 250, 10_000
dt = T / steps

Z = rng.standard_normal((n_paths, steps))
log_increments = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
S = S0 * np.exp(np.cumsum(log_increments, axis=1))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(S[:50].T, lw=0.7)
ax[0].set_title("50 simulated 1-year price paths")
ax[0].set_xlabel("trading day")
ax[1].hist(S[:, -1], bins=80)
ax[1].set_title("Distribution of price after 1 year")
plt.show()

ST = S[:, -1]
print(f"mean final price  : {ST.mean():.2f} (theory {S0 * np.exp(mu * T):.2f})")
print(f"median final price: {np.median(ST):.2f} (theory {S0 * np.exp((mu - 0.5 * sigma**2) * T):.2f})")
print(f"P(loss after 1y)  : {np.mean(ST < S0):.3f}")

TODO 4. Why is the mean final price above the median? Change `sigma` to 0.60 (Bitcoin-like volatility) and rerun. What happens to the probability of a loss, and to the gap between mean and median?

> your answer

## 5. Loops vs vectorization (why NumPy matters)

In [ ]:
def pi_loop(n, seed=0):
    import random
    random.seed(seed)
    inside = 0
    for _ in range(n):
        x, y = random.random(), random.random()
        if x * x + y * y <= 1.0:
            inside += 1
    return 4 * inside / n

n = 2_000_000
t0 = time.perf_counter(); pi_loop(n); t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); mc_pi(n, rng); t_vec = time.perf_counter() - t0
print(f"pure Python loop: {t_loop:.3f}s | NumPy vectorized: {t_vec:.3f}s | speedup ~{t_loop / t_vec:.0f}x")

TODO 5. Note the speedup. In Q18 you'll write the same thing in C++ and compare all three.

## Before you close this notebook
Write 3 lines in `notes/quant.md`:
1. What I learned today
2. What confused me
3. One question to ask